In [1]:
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

### Results / Training / Testing

This cell documents the image caption generation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.

```bash
pip install -r requirements.txt
```


In [2]:
%%writefile requirements.txt

nltk>=3.0.0
notebook>=7.0.0
seaborn
torch>=2.0.0
torchvision
tqdm>=4.0.0

Writing requirements.txt


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: jupyter-server
    Found existing installation: jupyter_server 2.12.5
    Uninstalling jupyter_server-2.12.5:
      Successfully uninstalled jupyter_server-2.12.5
  Attempting uninstall: jupyter-lsp
    Found existing installation: jupyter-lsp 1.5.1
    Uninstalling jupyter-lsp-1.5.1:
      Successfully uninstalled jupyter-lsp-1.5.1
  Attempting uninstall: jupyterlab
    Found existing installation: jupyterlab 3.6.8
    Uninstalling jupyterlab-3.6.8:
      Successfully uninstalled jupyterlab-3.6.8
  Attempting uninstall: notebook
    Found existing installation: notebook 6.

### Dataset / Run / Segmentation

This cell documents the image caption generation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [4]:
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt  # Kaggle上显示图片需要

# Note on Dataset, Data. Refer to the archived source for the detailed original comment.
IMAGE_CAPTION_ROOT = Path("/kaggle/input/datasets/adityajn105/flickr8k")
KARPATHY_ROOT = Path("/kaggle/input/datasets/shtvkumar/karpathy-splits")

json_path = KARPATHY_ROOT / "dataset_flickr8k.json"

with open(json_path, "r") as f:
    data = json.load(f)

print("First 2 images:")
for i in range(2):
    image_data = data["images"][i]
    img_path = IMAGE_CAPTION_ROOT / "Images" / image_data["filename"]
    image = Image.open(img_path)
    
    print(f"Image {i} ({image_data['filename']}, {image.size[0]}px x {image.size[1]}px):")
    print("Captions:")
    for caption in image_data["sentences"]:
        print(f"  - {caption['raw']}")
    
    # image.show()在Kaggle网页端无效，改用matplotlib显示
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Image: {image_data['filename']}")
    plt.show()
    print()

First 2 images:
Image 0 (2513260012_03d33305cf.jpg, 500px x 399px):
Captions:
  - A black dog is running after a white dog in the snow .
  - Black dog chasing brown dog through snow
  - Two dogs chase each other across the snowy ground .
  - Two dogs play together in the snow .
  - Two dogs running through a low lying body of water .


<Figure size 600x600 with 1 Axes>


Image 1 (2903617548_d3e38d7f88.jpg, 300px x 500px):
Captions:
  - A little baby plays croquet .
  - A little girl plays croquet next to a truck .
  - The child is playing croquette by the truck .
  - The kid is in front of a car with a put and a ball .
  - The little boy is playing with a croquet hammer and ball beside the car .


<Figure size 600x600 with 1 Axes>

### Training / Testing / Validation

This cell documents the image caption generation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [5]:
from collections import Counter
from tqdm import tqdm

min_word_freq = 5  # 最小词频，用于过滤掉不常用的词，减小词表大小


def build_vocab(karpathy_data: dict) -> dict[str, int]:
    """从Karpathy数据集构建词表

    Args:
        karpathy_data (dict): 包含图像标注数据的字典

    Returns:
        dict[str, int]: 词表，键为词，值为词在表中的索引
    """
    # 统计每个词的出现次数
    counter = Counter[str]()
    for image_data in tqdm(karpathy_data["images"], desc="Building vocab"):
        for caption in image_data["sentences"]:
            counter.update(caption["tokens"])

    words = [w for w in counter if counter[w] >= min_word_freq]
    words.sort(key=lambda x: counter[x], reverse=True)

    print(
        f"Vocab: total {len(counter)} words, clamped to {len(words)} words with "
        f"min_freq={min_word_freq}"
    )

    # Note on Create. Refer to the archived source for the detailed original comment.
    word_map = {}

    word_map["<pad>"] = 0
    word_map["<start>"] = 1
    word_map["<end>"] = 2
    word_map["<unk>"] = 3

    for w in words:
        word_map[w] = len(word_map)

    return word_map


def encode_captions(karpathy_data: dict, word_map: dict[str, int]):
    """Encode captions in Karpathy dataset to indices in word_map.

    Args:
        karpathy_data (dict): 包含图像标注数据的字典
        word_map (dict[str, int]): 词表，键为词，值为词在表中的索引

    Returns:
        list[dict]: 编码后的数据，每个元素是一个字典，包含图像文件名和编码后的句子列表。
    """
    # if max_len < 0:
    #     for image_data in karpathy_data["images"]:
    #         for caption in image_data["sentences"]:
    #             max_len = max(max_len, len(caption["tokens"]))
    #     max_len += 2  # Include <start> and <end>

    token_start = word_map["<start>"]
    token_end = word_map["<end>"]
    token_unk = word_map["<unk>"]  # Unknown
    encoded_data = []
    for image_data in karpathy_data["images"]:
        obj = {
            "filename": image_data["filename"],
            "split": image_data["split"],
            "sentences": [],
        }
        for caption in image_data["sentences"]:
            sentence = [token_start]
            for w in caption["tokens"]:
                sentence.append(word_map.get(w, token_unk))
            sentence.append(token_end)
            obj["sentences"].append(sentence)
        encoded_data.append(obj)
    return encoded_data

### Run

This cell documents the image caption generation experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [6]:
# Note on Save, Define. Refer to the archived source for the detailed original comment.
WORKING_ROOT = Path("/kaggle/working")

# 构建词表
word_map = build_vocab(data)
print(f"Total words in vocab (including special tokens): {len(word_map)}")
print(f"First 10 words: {', '.join(list(word_map.keys())[4:14])}")

# Note on Save. Refer to the archived source for the detailed original comment.
vocab_path = WORKING_ROOT / "vocab.json"
with open(vocab_path, "w") as f:
    json.dump(word_map, f)
print(f"✅ Vocab saved to {vocab_path}")

# Map caption tokens to indices
encoded_data = encode_captions(data, word_map)
encoded_path = WORKING_ROOT / "captions_encoded.json"
with open(encoded_path, "w") as f:
    json.dump(encoded_data, f)
print(f"✅ Encoded captions saved to {encoded_path}")

Building vocab: 100%|██████████| 8000/8000 [00:00<00:00, 110140.92it/s]


Vocab: total 8383 words, clamped to 2941 words with min_freq=5
Total words in vocab (including special tokens): 2945
First 10 words: a, in, the, on, is, and, dog, with, man, of
✅ Vocab saved to /kaggle/working/vocab.json
✅ Encoded captions saved to /kaggle/working/captions_encoded.json
